# Setup and data loading

The question: can a vendor-derived signal predict six-month credit deterioration if the joins only use information that was published at the time?

Discovery data is read from `data/` (or `TAKEHOME_DATA_DIR`). The holdout is only loaded further down, in the Holdout tests section, from `TAKEHOME_HOLDOUT_DIR` (default `../holdout`).

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, statsmodels.api as sm
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from takehome.analysis import *
from takehome.helpers import load_pack
D = prepare(os.environ.get("TAKEHOME_DATA_DIR", ROOT / "data"))
ev, out, iss, exits = D["events_clean"], D["outcomes"], D["issuers"], D["exits"]
cut = clip_points(ev, FEATURES)
dv = pit_sample(ev, out)
print("discovery: events", len(D["events"]), "-> after dedup/canary removal", len(ev), "| outcomes", out.shape, "| issuers", len(iss), "| exits", len(exits))

discovery: events 1285 -> after dedup/canary removal 1253 | outcomes (19584, 4) | issuers 421 | exits 24


# Observations

Before running any tests I checked the discovery data for duplicates, `-999` placeholders, mixed timezones, late publication and issuers missing from the outcomes. The numbers are printed below.

In [2]:
raw = D["events"]
feats = [c for c in raw if c.startswith("f_")]
print("rows removed by dedup/canary:", D["dedup_removed"], "| duplicate event_ids in raw:", int(raw.event_id.duplicated().sum()))
print("share of feature cells == -999 (raw):", round(float((raw[feats] == -999).mean().mean()), 4), "-> set to NaN")
print("event_tz values:", sorted(raw.event_tz.unique()), "-> event_time converted to UTC")
lag = (ev.published_at - ev.event_time_utc).dt.days
print("publication lag (days) buckets:", lag.groupby(pd.cut(lag, [-1, 10, 30, 60, 130])).size().to_dict())
in_month = ev.published_at.dt.strftime("%Y-%m") == ev.month
print(f"events published inside their event month: {in_month.mean():.1%}; the rest arrive 60-130 days late (revisions) and cannot be used at event month")
print("issuers with events but no outcome rows:", len(set(ev.issuer_id) - set(out.issuer_id)), "| of which exited:", len((set(ev.issuer_id) - set(out.issuer_id)) & set(exits.issuer_id)))
print("deterioration rate: all issuer-months", round(out.deterioration_6m.mean(), 3), "| issuer-months with a usable event", round(dv.deterioration_6m.mean(), 3))
print("point-in-time-valid discovery sample:", len(dv), "events,", dv.issuer_id.nunique(), "issuers")

rows removed by dedup/canary: 32 | duplicate event_ids in raw: 31
share of feature cells == -999 (raw): 0.0015 -> set to NaN
event_tz values: ['America/New_York', 'Asia/Tokyo', 'Europe/London'] -> event_time converted to UTC
publication lag (days) buckets: {Interval(-1, 10, closed='right'): 702, Interval(10, 30, closed='right'): 0, Interval(30, 60, closed='right'): 0, Interval(60, 130, closed='right'): 464}
events published inside their event month: 57.9%; the rest arrive 60-130 days late (revisions) and cannot be used at event month
issuers with events but no outcome rows: 12 | of which exited: 12
deterioration rate: all issuer-months 0.148 | issuer-months with a usable event 0.441
point-in-time-valid discovery sample: 696 events, 338 issuers


# Pre-registered hypotheses

I locked these two in the portal before the holdout was released. Both predict that a higher value means more deterioration, with a minimum effect of 0.05 per 1 SD. To pass, the one-sided p has to be below 0.025 (Bonferroni for two tests).

- H1: `f_14`, mechanism liquidity. A higher value would mean tighter funding or cash strain, so deterioration within six months is more likely.
- H2: `f_39`, mechanism operational. A higher value would mean weaker operating performance, which wears down credit quality over six months.

Test plan for both: take holdout events published inside their event month, match them to `deterioration_6m` for the same issuer and month, and fit a linear probability model on the standardized feature (clipped at the discovery 1st and 99th percentiles) with issuer-clustered errors.

Falsifier for both: the holdout slope per 1 SD is below 0.05, or not positive, or the one-sided clustered p is above 0.025.

The mechanism labels are just my best story after looking at the data. The features are anonymised, so I can't check them.

# Cheap first-pass tests

Discovery data only. I screened all 45 features on the sample of events published in their own month (slope per 1 SD, issuer-clustered errors, Benjamini-Hochberg over the 45), and then looked at how much the choice of join changes the answer.

In [3]:
rows = []
for f in FEATURES:
    r = lpm(dv, f, cut); rows.append((f, r["n"], r["estimate"] * 100, r["p_two_sided"]))
scr = pd.DataFrame(rows, columns=["feature", "n", "pp_per_sd", "p"]).set_index("feature")
scr["q_bh"] = sm.stats.multipletests(scr.p, method="fdr_bh")[1]
print(scr.reindex(scr.pp_per_sd.abs().sort_values(ascending=False).index).head(8).round(4))
# join sensitivity: naive event-month join (ignores publication time) vs point-in-time
naive = ev.merge(out, on=["issuer_id", "month"])
print("\nslope (pp per SD) naive join vs point-in-time:", {f: (round(lpm(naive, f, cut)["estimate"] * 100, 1), round(lpm(dv, f, cut)["estimate"] * 100, 1)) for f in ["f_08", "f_14", "f_39"]})

           n  pp_per_sd       p    q_bh
feature                                
f_14     692    32.5869  0.0000  0.0000
f_39     696     5.2191  0.0041  0.0921
f_05     696     4.7557  0.0088  0.1264
f_06     695     4.5746  0.0145  0.1264
f_25     695    -4.4737  0.0168  0.1264
f_15     696    -4.1811  0.0148  0.1264
f_32     695     3.6620  0.0538  0.3459
f_03     695    -2.9217  0.1070  0.5663

slope (pp per SD) naive join vs point-in-time: {'f_08': (18.8, -2.4), 'f_14': (31.1, 32.6), 'f_39': (4.1, 5.2)}


# Holdout tests

Each test was run once, exactly as pre-registered. The holdout events are new: no event IDs or issuer-months overlap with discovery.

In [4]:
HOLD = Path(os.environ.get("TAKEHOME_HOLDOUT_DIR", ROOT.parent / "holdout"))
H = prepare(HOLD)
hev = H["events_clean"]
hv = pit_sample(hev, H["outcomes"])
cut2 = {f: cut[f] for f in LOCKED.values()}
print("holdout: events", len(H["events"]), "-> clean", len(hev), "| published in event month and matched:", len(hv), "| issuers", hv.issuer_id.nunique(), "| deterioration rate", round(hv.deterioration_6m.mean(), 3))
print("overlapping issuer-months with discovery sample:", len(set(zip(dv.issuer_id, dv.month)) & set(zip(hv.issuer_id, hv.month))))
HOLDOUT = {}
for h, f in LOCKED.items():
    r = lpm(hv, f, cut2); r["decision"] = verdict(r); HOLDOUT[h] = r
    print(h, f, f"slope/SD={r['estimate']:.4f}  95% CI=[{r['ci'][0]:.4f}, {r['ci'][1]:.4f}]  one-sided p={r['p_one_sided']:.3g}  n={r['n']}  ->", r["decision"])
print("\nRobustness (same plan, holdout):")
exited = set(H["exits"].issuer_id); hx = with_exits_as_deteriorated(hev, H["outcomes"], H["exits"])
ISS = H["issuers"][["issuer_id", "size_bucket"]]
for h, f in LOCKED.items():
    uniq = hv.groupby(["issuer_id", "month"], as_index=False)[[f, "deterioration_6m"]].mean()
    print(h, f, "| no clipping", round(lpm(hv, f)["estimate"], 4), "| one row/issuer-month", round(lpm(uniq, f, cut2)["estimate"], 4),
          "| drop exited", round(lpm(hv[~hv.issuer_id.isin(exited)], f, cut2)["estimate"], 4),
          "| exited defaulters counted as deteriorated", round(lpm(hx, f, cut2)["estimate"], 4))
    print("   halves", {("early" if k else "late"): round(lpm(g, f, cut2)["estimate"], 3) for k, g in hv.groupby(hv.month < "2022-07")},
          "| size", {k: round(lpm(g, f, cut2)["estimate"], 3) for k, g in hv.merge(ISS, on="issuer_id").groupby("size_bucket")},
          "| source", {k: round(lpm(g, f, cut2)["estimate"], 3) for k, g in hv.groupby("source_class")})

holdout: events 3000 -> clean 2947 | published in event month and matched: 1398 | issuers 396 | deterioration rate 0.449
overlapping issuer-months with discovery sample: 0
H1 f_14 slope/SD=0.3185  95% CI=[0.3004, 0.3366]  one-sided p=3.7e-260  n=1395  -> keep
H2 f_39 slope/SD=0.0674  95% CI=[0.0387, 0.0962]  one-sided p=2.09e-06  n=1397  -> keep

Robustness (same plan, holdout):
H1 f_14 | no clipping 0.3153 | one row/issuer-month 0.3276 | drop exited 0.3194 | exited defaulters counted as deteriorated 0.3097


   halves {'late': 0.321, 'early': 0.315} | size {'large': 0.33, 'mid': 0.334, 'small': 0.291} | source {'8k_filing': 0.323, 'news_wire': 0.329, 'ratings_action': 0.3, 'research_note': 0.324}
H2 f_39 | no clipping 0.0677 | one row/issuer-month 0.0637 | drop exited 0.0673 | exited defaulters counted as deteriorated 0.0685
   halves {'late': 0.045, 'early': 0.091} | size {'large': 0.076, 'mid': 0.039, 'small': 0.087} | source {'8k_filing': 0.064, 'news_wire': 0.076, 'ratings_action': 0.069, 'research_note': 0.063}


In [5]:
print("Adjusted for controls (robustness, holdout): slope per 1 SD [95% CI] with fixed effects")
CTRL = {"none": (), "month": ("month",), "sector": ("sector",), "size": ("size_bucket",), "source": ("source_class",), "all four": ("month", "sector", "size_bucket", "source_class")}
ADJ = {}
for h, f in LOCKED.items():
    row = {}
    for name, c in CTRL.items():
        r = lpm_adjusted(hv, H["issuers"], f, cut2, c) if c else lpm(hv, f, cut2)
        row[name] = (round(r["estimate"], 4), [round(x, 3) for x in r["ci"]]); ADJ[h] = r if name == "all four" else ADJ.get(h, r)
    print(h, f, row)
print("\nsame check on discovery (sector only):", {f: round(lpm_adjusted(dv, D["issuers"], f, cut2, ("sector",))["estimate"], 4) for f in LOCKED.values()})

Adjusted for controls (robustness, holdout): slope per 1 SD [95% CI] with fixed effects
H1 f_14 {'none': (0.3185, [0.3, 0.337]), 'month': (0.3205, [0.302, 0.339]), 'sector': (0.3181, [0.299, 0.337]), 'size': (0.3184, [0.3, 0.337]), 'source': (0.3186, [0.3, 0.337]), 'all four': (0.3202, [0.301, 0.339])}


H2 f_39 {'none': (0.0674, [0.039, 0.096]), 'month': (0.068, [0.039, 0.097]), 'sector': (0.0308, [-0.006, 0.068]), 'size': (0.0673, [0.039, 0.096]), 'source': (0.0676, [0.039, 0.096]), 'all four': (0.0335, [-0.003, 0.07])}

same check on discovery (sector only): {'f_14': 0.324, 'f_39': 0.0237}


# Ways I might be fooling myself

A few things worry me. I searched 45 features, so something will look good by chance (the permutation test below checks this). A real predictor should only work at the right horizon, so I check timing. `f_14` separates the outcome almost perfectly, which is hard to believe. And events cluster where deterioration happens, while defaulted issuers are missing from the outcomes.

In [6]:
rng = np.random.default_rng(0)
X = dv[FEATURES].copy(); X = (X - X.mean()) / X.std(); yv = dv.deterioration_6m.values.astype(float)
def slopes(y):
    m = X.notna().values; Xc = np.where(m, X.values - np.nanmean(X.values, 0), 0)
    return (Xc * (y - y.mean())[:, None]).sum(0) / (Xc ** 2).sum(0) * 100
obs = slopes(yv); mx = []
for _ in range(200):
    yp = dv.assign(y=yv).groupby("month").y.transform(lambda s: rng.permutation(s.values)).values; mx.append(np.abs(slopes(yp)).max())
mx = np.array(mx)
print("null max |slope| 95th pct: %.2f pp (max %.2f)" % (np.percentile(mx, 95), mx.max()))
for f in ["f_14", "f_39"]:
    s = obs[FEATURES.index(f)]; print(f, "discovery slope %.2f pp -> family-wise permutation p = %.3f" % (s, (mx >= abs(s)).mean()))
oo = out.set_index(["issuer_id", "month"]).deterioration_6m
print("\ntiming placebo (pp per SD at other outcome-month offsets):")
for k in (-6, -3, 0, 3, 6, 12):
    d = dv.copy(); m = pd.PeriodIndex(d.month, freq="M") + k
    d["deterioration_6m"] = oo.reindex(pd.MultiIndex.from_arrays([d.issuer_id, m.strftime("%Y-%m")])).values
    print("  offset %+d" % k, {f: round(lpm(d.dropna(subset=["deterioration_6m"]), f, cut)["estimate"] * 100, 1) for f in ("f_14", "f_39")})
q = pd.qcut(dv.f_14, 10, duplicates="drop"); print("\nf_14 decile deterioration rates:", dv.groupby(q).deterioration_6m.mean().round(2).tolist())
print("rate with an event vs without (all issuer-months):", out.assign(has=out.set_index(["issuer_id", "month"]).index.isin(ev.set_index(["issuer_id", "month"]).index)).groupby("has").deterioration_6m.mean().round(3).to_dict())

null max |slope| 95th pct: 6.28 pp (max 7.70)
f_14 discovery slope 32.31 pp -> family-wise permutation p = 0.000
f_39 discovery slope 5.22 pp -> family-wise permutation p = 0.275

timing placebo (pp per SD at other outcome-month offsets):
  offset -6 {'f_14': 1.2, 'f_39': 1.4}
  offset -3 {'f_14': 1.0, 'f_39': 0.9}
  offset +0 {'f_14': 32.6, 'f_39': 5.2}
  offset +3 {'f_14': -1.2, 'f_39': -0.1}
  offset +6 {'f_14': -1.1, 'f_39': -1.0}
  offset +12 {'f_14': 0.0, 'f_39': -0.2}

f_14 decile deterioration rates: [0.2, 0.14, 0.13, 0.09, 0.09, 0.09, 0.67, 1.0, 1.0, 1.0]
rate with an event vs without (all issuer-months): {False: 0.131, True: 0.437}


What the output above says:

- `f_14` survives the permutation test over all 45 features and only predicts the same-month label. But the top three deciles are 100% deteriorated, and no real credit signal is that clean. It could be a planted effect in synthetic data, or something recorded alongside the label, and I can't tell which. The holdout replicates it, so it isn't noise, but that doesn't rule out leakage.
- `f_39` was weak in discovery (family-wise permutation p of about 0.27). It passes the locked holdout test, but sector controls roughly halve its effect in both samples, so I don't trust it.
- Events cluster where deterioration happens (about 44% versus 13% without an event), so the findings only apply to issuer-months that have an event.
- Defaulted issuers have no outcome rows. Counting them as deteriorated barely changes the estimates (see the robustness cell in the holdout section).
- I picked the mechanism labels after seeing the data.

# Conclusions

H1 (`f_14`, liquidity) is supported. The holdout slope is about +0.32 per SD, far above the 0.05 bar, and it didn't move under any check I tried.

H2 (`f_39`, operational) is inconclusive. It passes the locked test (+0.067, 95% CI roughly 0.04 to 0.10), but with sector fixed effects it falls to about +0.03 and stops being significant, and the same happens in discovery. Most of what `f_39` picks up is which sectors deteriorate more, so I'm not claiming it as a real signal.

Each hypothesis got one pre-registered test on a holdout that shares nothing with discovery. The near-perfect separation of `f_14` is the main reason to be careful about how large its effect looks.

In [7]:
def gate(h):  # decision rule: the locked test must pass AND still pass (>= 0.05, one-sided p < 0.025) with all four fixed effects
    a = ADJ[h]; return ("keep", "") if (HOLDOUT[h]["decision"] == "keep" and a["estimate"] >= MIN_EFFECT and a["p_one_sided"] < ALPHA) else ("inconclusive", "concentration")
hyp = [dict(hypothesis_id=h, feature=LOCKED[h], estimate=round(r["estimate"], 6), ci=[round(c, 6) for c in r["ci"]],
            p_value=float(r["p_one_sided"]), n=r["n"], decision=gate(h)[0], kill_reason=gate(h)[1],
            locked_test_passed=r["decision"] == "keep", adjusted_estimate=round(ADJ[h]["estimate"], 6),
            adjusted_ci=[round(c, 6) for c in ADJ[h]["ci"]]) for h, r in HOLDOUT.items()]
checks = dict(dedup_rows_removed=int(D["dedup_removed"] + H["dedup_removed"]),
              sentinel_handling="-999 treated as missing (NaN) before clipping and standardizing; for duplicate event_ids I kept the copy with the fewest missing features",
              timezone_normalized=True, exits_included=True,
              exits_note="primary tests only use issuers that have outcome rows; as a robustness check, defaulted and distressed-exchange issuers without outcome rows were counted as deteriorated",
              point_in_time="only events published inside their event month are used; later revisions are excluded",
              issuer_id_normalized="issuer ids lowercased and stripped before every join")
(ROOT / "submit" / "results.json").write_text(json.dumps(dict(hypotheses=hyp, data_checks=checks), indent=2))
print(json.dumps(dict(hypotheses=hyp, data_checks=checks), indent=2))

{
  "hypotheses": [
    {
      "hypothesis_id": "H1",
      "feature": "f_14",
      "estimate": 0.318479,
      "ci": [
        0.300352,
        0.336606
      ],
      "p_value": 3.695563751478017e-260,
      "n": 1395,
      "decision": "keep",
      "kill_reason": "",
      "locked_test_passed": true,
      "adjusted_estimate": 0.32022,
      "adjusted_ci": [
        0.301043,
        0.339397
      ]
    },
    {
      "hypothesis_id": "H2",
      "feature": "f_39",
      "estimate": 0.06745,
      "ci": [
        0.038722,
        0.096178
      ],
      "p_value": 2.0940797025595638e-06,
      "n": 1397,
      "decision": "inconclusive",
      "kill_reason": "concentration",
      "locked_test_passed": true,
      "adjusted_estimate": 0.033536,
      "adjusted_ci": [
        -0.003394,
        0.070466
      ]
    }
  ],
  "data_checks": {
    "dedup_rows_removed": 85,
    "sentinel_handling": "-999 treated as missing (NaN) before clipping and standardizing; for duplicate even